# EDA Phase 4: Symptom Behavior Across Menstrual Phases

**Project:** CycleInsight  
**Dataset:** `datasets/cleaned_dataset.csv`  
**Grain:** One row per day-level record. Repeated records from the same participant are not independent.  
**Scope:** Descriptive analysis of symptom burden and symptom prevalence across the recorded menstrual phases. No medical, causal, or predictive claims are made.


## Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path


## Configuration

In [ ]:
# Keep paths relative to the notebook so it can be run from the project root or notebooks folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "datasets" / "cleaned_dataset.csv"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PHASE_COL = "phase"
SYMPTOM_COLS = [
    "headaches", "cramps", "sorebreasts", "fatigue", "sleepissue",
    "moodswing", "stress", "foodcravings", "indigestion", "bloating",
]
PHASE_ORDER = ["Menstrual", "Follicular", "Fertility", "Luteal"]
ABSENT_RESPONSES = {"not at all", "none", "no", "0"}

sns.set_theme(style="whitegrid", context="notebook")


## Data Loading

In [ ]:
# Load the cleaned day-level dataset once for all analyses.
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} records and {df.shape[1]} columns from {DATA_PATH.name}.")
df.head()


## Validation

In [ ]:
# Check that all fields used below exist and inspect missingness and phase counts.
required_cols = [PHASE_COL, *SYMPTOM_COLS]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Required columns are missing: {missing_cols}")

validation = pd.DataFrame({
    "missing_count": df[required_cols].isna().sum(),
    "missing_percent": (df[required_cols].isna().mean() * 100).round(2),
})
print(f"Records: {len(df):,}; unique participants: {df['id'].nunique() if 'id' in df else 'not available'}")
display(validation)
display(df[PHASE_COL].value_counts(dropna=False).rename("records").to_frame())


## Symptom Burden Construction

In [ ]:
# Presence is any recorded symptom response other than a recognized absence response.
# Missing symptom answers stay missing; burden therefore counts observed symptoms present.
symptom_flags = pd.DataFrame(index=df.index)
for col in SYMPTOM_COLS:
    normalized = df[col].astype("string").str.strip().str.lower()
    symptom_flags[col] = normalized.notna() & ~normalized.isin(ABSENT_RESPONSES)
    symptom_flags.loc[normalized.isna(), col] = pd.NA
    symptom_flags[col] = symptom_flags[col].astype("Int64")

analysis_df = df[[PHASE_COL]].copy()
analysis_df[SYMPTOM_COLS] = symptom_flags
analysis_df["symptom_burden"] = symptom_flags.sum(axis=1, min_count=1)

print("Symptom burden counts observed-present symptoms on each record.")
print(f"Records with any missing symptom answer: {symptom_flags.isna().any(axis=1).sum():,}")


## Symptom Burden Analysis

In [ ]:
# Summarize the record-level burden distribution overall and by phase.
burden_summary = analysis_df["symptom_burden"].describe().round(2).to_frame("symptom_burden")
phase_burden = (
    analysis_df.dropna(subset=[PHASE_COL])
    .groupby(PHASE_COL)["symptom_burden"]
    .agg(["count", "mean", "median", "std"])
    .reindex(PHASE_ORDER)
    .round(2)
)
display(burden_summary)
display(phase_burden)


## Symptom Prevalence by Phase

In [ ]:
# Divide present responses by non-missing responses for each symptom within each phase.
phase_prevalence = (
    analysis_df.dropna(subset=[PHASE_COL])
    .groupby(PHASE_COL)[SYMPTOM_COLS]
    .mean()
    .reindex(PHASE_ORDER)
    .mul(100)
)
phase_prevalence.index.name = "phase"
display(phase_prevalence.round(1))


## Visualizations

In [ ]:
# Save all requested visual summaries as standalone PNG files.
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(analysis_df["symptom_burden"].dropna(), bins=range(0, len(SYMPTOM_COLS) + 2),
             discrete=True, color="#4C78A8", ax=ax)
ax.set(title="Symptom Burden Across Records", xlabel="Number of symptoms present", ylabel="Record count")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "eda4_symptom_burden_histogram.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
plot_data = analysis_df.dropna(subset=[PHASE_COL, "symptom_burden"])
sns.boxplot(data=plot_data, x=PHASE_COL, y="symptom_burden", order=PHASE_ORDER,
            color="#9ecae1", showfliers=False, ax=ax)
ax.set(title="Symptom Burden by Menstrual Phase", xlabel="Menstrual phase", ylabel="Symptoms present per record")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "eda4_symptom_burden_by_phase.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(phase_prevalence, annot=True, fmt=".1f", cmap="YlOrRd", vmin=0, vmax=100,
            cbar_kws={"label": "Records with symptom present (%)"}, ax=ax)
ax.set(title="Symptom Prevalence by Menstrual Phase", xlabel="Symptom", ylabel="Menstrual phase")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "eda4_symptom_prevalence_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

# Rank symptoms within phase and chart the three highest prevalences per phase.
top_n = 3
top_symptoms = (
    phase_prevalence.rename_axis("phase").reset_index()
    .melt(id_vars="phase", var_name="symptom", value_name="prevalence_percent")
    .sort_values(["phase", "prevalence_percent"], ascending=[True, False])
    .groupby("phase", sort=False)
    .head(top_n)
)
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=top_symptoms, x="prevalence_percent", y="symptom", hue="phase",
            hue_order=PHASE_ORDER, ax=ax)
ax.set(title=f"Top {top_n} Symptoms by Phase", xlabel="Records with symptom present (%)", ylabel="Symptom")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "eda4_top_symptoms_by_phase.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Figures saved to {FIGURE_DIR}")


## Findings

### Key Findings

- Symptom burden averaged **6.9 observed symptoms per record** (median **8**, range **0–10**).
- Menstrual-phase records had the highest mean burden (**8.1**); the other phase means ranged from **6.6 to 6.7**.
- Fatigue and stress were among the most prevalent symptoms in each phase. In menstrual-phase records, headaches (78.5%) and cramps (78.0%) were also frequently reported.
- These are record-weighted descriptions. Participants contributed repeated day-level records, so results do not represent independent observations or participant-level averages.

### Data Quality Notes

- One sorebreasts response is missing; other requested symptom fields and the phase field have no missing values in the inspected dataset.
- Missing symptom responses are left unclassified and excluded from the relevant prevalence denominator. Burden is the number of observed-present symptoms, so a missing answer may make a record's count lower than its complete symptom count.
- The analysis uses the recorded `phase` labels and treats any non-missing response other than “Not at all” as symptom presence.

### Recommended Next Step

Compare these record-level patterns with participant-level phase summaries to check whether the observed differences persist when each participant contributes equally.
